# S23DR 2026 — RoofWireframeNet Training

**Runtime**: GPU (Runtime → Change runtime type → T4 GPU)  
**Expected time**: ~3.6 min/epoch on T4, ~1.2 min/epoch on A100

Steps:
1. Check GPU
2. Install dependencies
3. Clone repo
4. Verify imports
5. Smoke test
6. Benchmark step time
7. (Optional) Restore checkpoint from Drive
8. Load dataset
9. Run training — saves `last.pt` every epoch, `best.pt` on improvement
10. Save checkpoint to Google Drive
11. Download checkpoint

In [ ]:
# ── 1. Check GPU ─────────────────────────────────────────────────────────────
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# ── 2. Install dependencies ──────────────────────────────────────────────────
!pip install -q datasets huggingface_hub scipy

In [ ]:
# ── 3. Clone repo ────────────────────────────────────────────────────────────
import os
if not os.path.exists('3d_building_construction'):
    !git clone https://github.com/12turtleships/3d_building_construction.git
%cd 3d_building_construction
!git pull
!git log --oneline -3

In [ ]:
# ── 4. Verify imports ────────────────────────────────────────────────────────
import sys
sys.path.insert(0, '.')
from s23dr.model import RoofWireframeNet, WireframeLoss, N_EDGE_CLASSES
from s23dr.data  import S23DRDataset, collate_fn

model = RoofWireframeNet(n_queries=64)
total = sum(p.numel() for p in model.parameters())
print(f'Model: {total:,} parameters')
print('OK')

In [ ]:
# ── 5. Smoke test: forward pass on GPU ──────────────────────────────────────
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using:', device)

model = RoofWireframeNet(n_queries=64).to(device)
loss_fn = WireframeLoss()

B, N = 4, 1024
xyz  = torch.randn(B, N, 3).to(device)
vf   = torch.rand(B, N).to(device)
nv   = torch.randint(0, 8, (B, N)).float().to(device)
msk  = torch.ones(B, N).to(device)
cid  = torch.randint(0, 10, (B, N)).to(device)
gv   = [torch.randn(31, 3) for _ in range(B)]
ge   = [torch.randint(0, 31, (32, 2)) for _ in range(B)]
gc   = [torch.randint(0, 10, (32,)) for _ in range(B)]

out    = model(xyz, vf, nv, msk, cid)
losses = loss_fn(out['pred_pos'], out['pred_conf'], out['edge_logits'], gv, ge, gc)
print('pred_pos:', tuple(out['pred_pos'].shape))
print('loss:    ', losses['loss'].item())
print('Smoke test passed!')

In [ ]:
# ── 6. Benchmark step time on GPU ────────────────────────────────────────────
import time
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)

for _ in range(3):
    out = model(xyz, vf, nv, msk, cid)
    loss_fn(out['pred_pos'], out['pred_conf'], out['edge_logits'], gv, ge, gc)['loss'].backward()
    opt.step(); opt.zero_grad()
if device == 'cuda': torch.cuda.synchronize()

times = []
for _ in range(10):
    t0 = time.time()
    out = model(xyz, vf, nv, msk, cid)
    loss_fn(out['pred_pos'], out['pred_conf'], out['edge_logits'], gv, ge, gc)['loss'].backward()
    opt.step(); opt.zero_grad()
    if device == 'cuda': torch.cuda.synchronize()
    times.append(time.time() - t0)

avg = sum(times) / len(times)
steps_per_epoch = 15892 // B
secs_per_epoch  = avg * steps_per_epoch
print(f'Step time:   {avg*1000:.0f} ms  (batch={B}, n_points=1024)')
print(f'Steps/epoch: {steps_per_epoch}')
print(f'Time/epoch:  {secs_per_epoch/60:.1f} min')
print(f'100 epochs:  {secs_per_epoch*100/3600:.1f} hrs')

In [ ]:
# ── 7. Version setup ─────────────────────────────────────────────────────────
# Each training run gets an auto-incremented version (v1, v2, …).
# Checkpoints are saved as  s23dr_v{N}_last.pt / s23dr_v{N}_best.pt  on Drive.
#
# To start a NEW run       → leave RESUME = False  (default)
# To continue latest run   → set   RESUME = True
# To resume specific run   → set   RESUME = True  and  VERSION = <number>

import re
import torch
from pathlib import Path
from google.colab import drive

if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')

DRIVE_DIR = Path('/content/drive/MyDrive')
RESUME    = False   # ← change to True to continue a previous run

existing = sorted(DRIVE_DIR.glob('s23dr_v*_last.pt'))
if existing:
    versions = [int(re.search(r'v(\d+)', p.stem).group(1)) for p in existing]
    latest   = max(versions)
    VERSION  = latest if RESUME else latest + 1
else:
    VERSION = 1
    RESUME  = False  # nothing to resume

VERSION = 4   # architecture changed (cross-attention decoder) — must retrain from scratch
DRIVE_LAST = str(DRIVE_DIR / f's23dr_v{VERSION}_last.pt')
DRIVE_BEST = str(DRIVE_DIR / f's23dr_v{VERSION}_best.pt')

# Print version history
print(f"{'Ver':>5}  {'File':>30}  {'Epoch':>6}  {'Val loss':>10}")
print("-" * 60)
for p in existing:
    v = int(re.search(r'v(\d+)', p.stem).group(1))
    try:
        ck = torch.load(p, map_location='cpu', weights_only=False)
        ep = ck.get('epoch', '?')
        vl = f"{ck.get('val_loss', float('nan')):.4f}"
    except Exception:
        ep, vl = '?', 'error'
    marker = ' ◀ resuming' if (RESUME and v == VERSION) else ''
    print(f"  v{v:<3}  {p.name:>30}  {str(ep):>6}  {vl:>10}{marker}")
print("-" * 60)
action = f"RESUME v{VERSION}" if RESUME else f"NEW RUN v{VERSION}"
print(f"▶ {action}")
print(f"  last → {DRIVE_LAST}")
print(f"  best → {DRIVE_BEST}")

In [ ]:
# ── 8. (Optional) Restore checkpoint from Drive ──────────────────────────────
# Controlled by RESUME and DRIVE_LAST set in the version-setup cell above.
# Skip this cell to always start from scratch.

import shutil
from pathlib import Path

LOCAL_DIR = Path('outputs/checkpoints')
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

if RESUME and Path(DRIVE_LAST).exists():
    shutil.copy(DRIVE_LAST, LOCAL_DIR / 'last.pt')
    print(f'Restored v{VERSION} checkpoint from {DRIVE_LAST}')
elif RESUME:
    print(f'RESUME=True but {DRIVE_LAST} not found — starting fresh as v{VERSION}.')
    RESUME = False
else:
    print(f'Starting fresh as v{VERSION}.')

In [ ]:
# ── 8. Load dataset ───────────────────────────────────────────────────────────
print('Loading train split (~15 k samples)…')
train_ds = S23DRDataset(split='train',      n_points=1024)
val_ds   = S23DRDataset(split='validation', n_points=1024)
print(f'Train: {len(train_ds)}  Val: {len(val_ds)}')

In [ ]:
# ── 10. Full training ────────────────────────────────────────────────────────
import time
import shutil
from pathlib import Path
from torch.utils.data import DataLoader
import torch.optim as optim
from google.colab import drive

if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')

EPOCHS       = 100
BATCH_SIZE   = 8
LR           = 1e-3
N_QUERIES    = 64
SAVE_EVERY_N = 5    # ← save a named epoch checkpoint every N epochs (0 = off)
                    #   100 epochs × 22 MB ÷ 5 = ~20 files = ~440 MB on Drive

CKPT_DIR  = Path('outputs/checkpoints')
CKPT_DIR.mkdir(parents=True, exist_ok=True)
LAST_CKPT = CKPT_DIR / 'last.pt'
BEST_CKPT = CKPT_DIR / 'best.pt'
# DRIVE_LAST, DRIVE_BEST, VERSION set by version-setup cell

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, collate_fn=collate_fn, drop_last=True,
                          pin_memory=(device=='cuda'))
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, collate_fn=collate_fn,
                          pin_memory=(device=='cuda'))

model     = RoofWireframeNet(n_queries=N_QUERIES).to(device)
loss_fn   = WireframeLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

start_epoch = 1
best_val    = float('inf')

if LAST_CKPT.exists():
    ckpt = torch.load(LAST_CKPT, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    start_epoch = ckpt['epoch'] + 1
    best_val    = ckpt.get('val_loss', float('inf'))
    print(f'Resumed v{VERSION} from epoch {ckpt["epoch"]} → continuing from {start_epoch}')
else:
    print(f'v{VERSION} — starting fresh  ({sum(p.numel() for p in model.parameters()):,} params on {device})')

def _save(local_path, epoch, val_loss):
    torch.save({'epoch': epoch, 'model': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict(),
                'val_loss': val_loss, 'version': VERSION}, local_path)

def _sync(local_path, drive_path):
    try:
        shutil.copy(local_path, drive_path)
    except Exception as e:
        print(f'  Warning: Drive sync failed: {e}')

LOG_EVERY = 200

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for step, batch in enumerate(train_loader):
        xyz, vf, nv, msk, cid = (batch['xyz'].to(device), batch['vote_frac'].to(device),
                                   batch['n_views'].to(device), batch['mask'].to(device),
                                   batch['class_id'].to(device))
        out  = model(xyz, vf, nv, msk, cid)
        loss = loss_fn(out['pred_pos'], out['pred_conf'], out['edge_logits'],
                       batch['gt_verts'], batch['gt_edges'], batch['gt_classes'])['loss']
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()
        if (step + 1) % LOG_EVERY == 0:
            print(f'  ep{epoch} step{step+1}/{len(train_loader)} loss={loss.item():.4f}')

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            out = model(batch['xyz'].to(device), batch['vote_frac'].to(device),
                        batch['n_views'].to(device), batch['mask'].to(device),
                        batch['class_id'].to(device))
            val_loss += loss_fn(out['pred_pos'], out['pred_conf'], out['edge_logits'],
                                batch['gt_verts'], batch['gt_edges'], batch['gt_classes'])['loss'].item()
    val_loss /= len(val_loader)
    scheduler.step()

    print(f'[v{VERSION}] Epoch {epoch:3d}/{EPOCHS} | Train: {epoch_loss/len(train_loader):.4f} '
          f'| Val: {val_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}')

    # Always save last (overwrites every epoch)
    _save(LAST_CKPT, epoch, val_loss)
    _sync(LAST_CKPT, DRIVE_LAST)

    # Named epoch checkpoint every N epochs
    if SAVE_EVERY_N > 0 and epoch % SAVE_EVERY_N == 0:
        epoch_drive = str(Path(DRIVE_LAST).parent / f's23dr_v{VERSION}_epoch{epoch:03d}.pt')
        _sync(LAST_CKPT, epoch_drive)
        print(f'  📌 Epoch checkpoint → s23dr_v{VERSION}_epoch{epoch:03d}.pt')

    # Save best
    if val_loss < best_val:
        best_val = val_loss
        _save(BEST_CKPT, epoch, val_loss)
        _sync(BEST_CKPT, DRIVE_BEST)
        print(f'  ✓ Best checkpoint saved → Drive (val={val_loss:.4f})')

print(f'Training finished. v{VERSION} best val loss: {best_val:.4f}')

In [ ]:
import torch
from pathlib import Path

ckpt_path = Path('outputs/checkpoints/last.pt')
print(f"Checking path: {ckpt_path.absolute()}")

if ckpt_path.exists():
    print("✅ Checkpoint file found.")
    try:
        checkpoint = torch.load(ckpt_path, map_location='cpu')
        print(f"Keys in checkpoint: {list(checkpoint.keys())}")
        if 'epoch' in checkpoint:
            print(f"Last saved epoch: {checkpoint['epoch']}")
        if 'model' in checkpoint:
            print("Model state_dict is present.")
    except Exception as e:
        print(f"❌ Error loading checkpoint: {e}")
else:
    print("❌ Checkpoint file NOT found at the expected location.")

# Also check the directory content
print(f"\nContent of outputs/checkpoints: {list(Path('outputs/checkpoints').glob('*'))}")

In [ ]:
from google.colab import drive
import os

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

print("Searching for ALL .pt files on Drive (this may take a moment)...")
!find /content/drive/MyDrive -name "*.pt"

In [ ]:
import os
print("--- Current Local Directory Structure ---")
!ls -R /content/3d_building_construction/outputs 2>/dev/null || echo "No 'outputs' directory found."

print("\n--- Root /content directory ---")
!ls /content

In [ ]:
# ── 11. Save checkpoint to Google Drive (manual backup) ──────────────────────
# The training loop already syncs every epoch. Run this if you need a manual save.
import shutil
from pathlib import Path
from google.colab import drive

if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')

for local, remote in [('outputs/checkpoints/last.pt', DRIVE_LAST),
                      ('outputs/checkpoints/best.pt', DRIVE_BEST)]:
    if Path(local).exists():
        shutil.copy(local, remote)
        print(f'Saved {local} → {remote}')
    else:
        print(f'Skipped {local} (not found)')

In [ ]:
# ── 11. Download checkpoint to local machine ─────────────────────────────────
from google.colab import files
files.download('outputs/checkpoints/best.pt')

In [ ]:
# ── Verify checkpoints on Drive ──────────────────────────────────────────────
import os, re, torch
from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive')
print(f"{'Ver':>5}  {'File':>32}  {'Epoch':>6}  {'Val loss':>10}  {'Size':>8}")
print("-" * 68)
for p in sorted(DRIVE_DIR.glob('s23dr_v*_*.pt')):
    try:
        ck = torch.load(p, map_location='cpu', weights_only=False)
        ep = ck.get('epoch', '?')
        vl = f"{ck.get('val_loss', float('nan')):.4f}"
        v  = ck.get('version', re.search(r'v(\d+)', p.stem).group(1))
    except Exception:
        ep, vl, v = '?', 'error', '?'
    size = f"{os.path.getsize(p)/1e6:.1f} MB"
    print(f"  v{str(v):<3}  {p.name:>32}  {str(ep):>6}  {vl:>10}  {size:>8}")